# RG3: 가스 → 미성형 순차 탐지

가스 최고 개발 후보(17개 중 8개 탐지·정상 오탐 6개) 다음에 공통 이력 기반 미성형 후보(8개 중 5개 탐지·정상 오탐 10개)를 적용한다.

- 공식 기존 RG3 RH 591샷: 정상 566, 가스 17, 미성형 8. 추가 가스 19개 코호트는 제외.
- 가스: 노트북 22/47/54/60의 고정 조합. 학습은 정상·가스만, 평가에서는 미성형까지 실제 예측한다. 기존 학습 내부 구간검증 임계값을 고정 재사용하며 외부 평가 라벨로 보정하지 않는다.
- 미성형: 노트북 151 fixed_OR(상대 온도변화 6피처 앙상블 + 동시변화 강도 4피처 로지스틱). 학습은 정상·미성형만. 기존 구간제외 예측을 재사용한다.
- 각 평가 구간은 학습에서 제외. 다만 이 후보들은 동일 자료를 반복 검토하며 선택했으므로 **미관측 최종 독립검증 성능이 아닌 개발 구간교차검증 결과**다. 구간제외는 미래 예측 검증과도 다르다.
- 가스 판정 샷을 먼저 제거하고 남은 샷만 미성형 판정한다. 정답을 보고 분기하지 않으며, 필터링 후 과거 이력을 다시 계산하지 않는다.
- 센서 이력은 과거 비라벨 센서값도 사용하지만 비라벨을 정상 정답으로 학습하지 않는다.
- 예전 미성형 F1 0.48 후보가 아니라, 라벨·비라벨에 같은 이력 계산식을 쓰는 최근 F1 0.435 후보를 결합한다.

아래 셀을 실행하면 두 단계와 최종 불량 여부 지표, 종류별 혼동행렬, 모든 불량의 판정 목록을 저장한다.

In [1]:
from pathlib import Path
import json, warnings
import numpy as np,pandas as pd
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import ExtraTreesClassifier
root=Path.cwd();tables=root/'reports/tables'
def source(name):
 n=json.loads((root/'notebooks'/name).read_text(encoding='utf-8'))
 return '\n'.join(''.join(c['source']) for c in n['cells'] if c['cell_type']=='code')
# Reuse original model definitions without modifying the source notebooks.
s22=source('22_rg3_nested_signal_ensemble.ipynb').split('scores=[];thresholds=[];foldlogs=[]')[0]
s22=s22.replace("if not (root/'data').exists():root=root.parent","").replace("assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='hayeon'","").replace('n_jobs=-1','n_jobs=2')
exec(s22)
official_count=len(d)
full=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
extra=full.loc[full['product'].eq('RG3')&~full.TimeStamp.isin(d.TimeStamp)]
assert len(extra)==8 and extra.Reason.str.contains('미성형',na=False).all()
d=pd.concat([d,extra],ignore_index=True)
y=np.r_[y,np.full(8,-1)];groups=d.comparison_block
assert len(d)==591 and (y==1).sum()==17 and (y==0).sum()==566
# -1 means shortshot: query only, NEVER admitted to gas training.
s47=source('47_rg3_innovation_seed_sensitivity.ipynb')
e=s47[s47.index('def expert('):s47.index('records=[];audit=[];rawrecords=[]')]
e=e.replace('scores=[[],[]]','scores=[[],[],[]]').replace('enumerate([7,101])','enumerate([42,7,101])')
memo={};exec(e)
gas_expert=expert
s54=source('54_rg3_pdf_thermal_confirm_gates.ipynb')
helpers=s54[s54.index('def local_residual('):s54.index("models=['two_tree_mean")]
exec(helpers);conditional_predict=predict
conditional_source=helpers[helpers.index('def predict('):]
mold=(d.Mold_Temperature_3+d.Mold_Temperature_4)/2
barrel=d[[f'Barrel_Temperature_{i}' for i in range(1,7)]].mean(axis=1)
F={'thermal':pd.DataFrame({
 'gap':barrel-mold,'side':d.Mold_Temperature_3-d.Mold_Temperature_4,
 'delta':(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'dev':(d.Mold_Temperature_3__dev10+d.Mold_Temperature_4__dev10)/2,
 'gap_delta':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'pressure_fill':d.Max_Injection_Pressure/d.Filling_Time.where(d.Filling_Time.abs()>1e-6)}),
 'pressure':pd.DataFrame({
 'dp_ds':d.Max_Injection_Pressure__delta1-d.Max_Switch_Over_Pressure__delta1,
 'dv':d.Max_Injection_Speed__delta1,'df':d.Filling_Time__delta1,
 'vp_vs':d.Max_Injection_Pressure__dev10-d.Max_Switch_Over_Pressure__dev10,
 'dc':d.Cushion_Position__delta1,
 'db_dm':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2})}
controls={'thermal':['Barrel_Temperature_5','Max_Injection_Speed','Filling_Time','Hopper_Temperature'],
 'pressure':['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3']}
for a in F.values():a.replace([np.inf,-np.inf],np.nan,inplace=True)
def table(name):
 z=pd.read_csv(tables/(name+'.csv'))
 return z.loc[z.scheme.eq('leave_one_block_out')]
basecuts=table('rg3_nested_ensemble_thresholds').query("model == 'max_union'").set_index('block')
innovation=table('rg3_innovation_seed_sensitivity_predictions').query("model == 'three_seed_mean'").groupby('block').first()
thermalcuts=table('rg3_pdf_confirm_gates_inner_candidates').query("gate == 'thermal_change_low_0.05'").set_index('block')
pressurecuts=table('rg3_pressure_seed_predictions').query("anchor == 'exploratory54' and model == 'pressure_fall_speed_rise_switch_steeper'").groupby(['block','configuration']).first()
dp=np.round(d.Max_Injection_Pressure__delta1.to_numpy(),2)
ds=np.round(d.Max_Switch_Over_Pressure__delta1.to_numpy(),2)
dv=np.round(d.Max_Injection_Speed__delta1.to_numpy(),2)
direction=(dp<0)&(dv>0)&(ds<dp)
rows=[];audit=[]
for b in blocks:
 ref=np.flatnonzero(groups.ne(b)&(y>=0));q=np.flatnonzero(groups.eq(b))
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q]) and (y[ref]>=0).all()
 baseline=pair_predict(ref,q).max(axis=1)>basecuts.loc[b,'threshold']
 memo={};innovation_score=gas_expert(ref,q).mean(axis=1)
 innovation_flag=(innovation_score>innovation.loc[b,'expert_threshold'])&(d.Barrel_Temperature_5__delta1.iloc[q].to_numpy()>innovation.loc[b,'delta_threshold'])
 # Thresholds are frozen inner-training calibration from previous notebooks.
 # Refit identical outer models on the identical ref rows, evaluate ALL 591 shots.
 exec(conditional_source);memo={}
 ts=predict(ref,q,'thermal')[:,1];tz=local_residual(ref,q,'thermal')[:,4]
 thermal_flag=(ts>thermalcuts.loc[b,'score_threshold'])&(tz<json.loads(thermalcuts.loc[b,'gate_thresholds'])[0])
 votes=[]
 for tag,seeds in [('seed42_7_101',[42,7,101]),('seed11_23_37',[11,23,37]),('seed13_29_53',[13,29,53])]:
  exec(conditional_source.replace('for seed in [42,7,101]:',f'for seed in {seeds}:'));memo={}
  ps=predict(ref,q,'pressure')[:,1]
  votes.append((ps>pressurecuts.loc[(b,tag),'threshold'])&direction[q])
 vote=np.sum(votes,axis=0)
 pr=baseline|innovation_flag|thermal_flag|(vote>=2)
 for k,i in enumerate(q):
  rows.append(dict(TimeStamp=d.TimeStamp.iloc[i],block=b,gas_prediction=int(pr[k]),baseline=int(baseline[k]),innovation=int(innovation_flag[k]),thermal=int(thermal_flag[k]),pressure_votes=int(vote[k])))
 audit.append(dict(block=b,gas_train=len(ref),gas_train_positive=int((y[ref]==1).sum()),test=len(q),test_train_timestamp_overlap=len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q])),shortshot_in_gas_train=int((y[ref]==-1).sum())))
 print('Completed gas full-cohort evaluation',b,flush=True)
gas=pd.DataFrame(rows)
old=table('rg3_pressure_seed_majority_predictions').query("anchor == 'exploratory54'")
check=gas.merge(old[['TimeStamp','prediction']].assign(TimeStamp=lambda z:pd.to_datetime(z.TimeStamp)),on='TimeStamp',validate='one_to_one')
assert len(check)==583 and (check.gas_prediction==check.prediction).all(), 'Gas model reproduction mismatch: stop, do not publish'
print('Original 583 gas predictions reproduced exactly; eight shortshots also genuinely scored.')
short=pd.read_csv(tables/'rg3_shortshot_151_predictions.csv',parse_dates=['TimeStamp'])
short=short.loc[short.model.eq('fixed_OR'),['TimeStamp','type','run_id','prediction','thermal','joint']].rename(columns={'prediction':'shortshot_prediction','thermal':'shortshot_thermal','joint':'shortshot_joint'})
out=gas.merge(short,on='TimeStamp',validate='one_to_one').sort_values('TimeStamp').reset_index(drop=True)
assert len(out)==591 and out.type.value_counts().to_dict()=={'정상':566,'가스':17,'미성형':8}
out['stage2_checked']=out.gas_prediction.eq(0)
out['stage2_flag']=out.stage2_checked&out.shortshot_prediction.eq(1)
out['final_defect']=out.gas_prediction.eq(1)|out.stage2_flag
out['predicted_type']=np.where(out.gas_prediction.eq(1),'가스',np.where(out.stage2_flag,'미성형','정상'))
# Historical feature windows were computed BEFORE filtering. Do not recompute
# windows after removing gas predictions, and never route with true labels.
def metric(name,truth,pred):
 tn,fp,fn,tp=confusion_matrix(truth,pred,labels=[0,1]).ravel()
 return dict(evaluation=name,shots=len(truth),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn) if tp+fn else 0,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0)
normal_or_gas=out.type.isin(['정상','가스']);normal_or_short=out.type.isin(['정상','미성형'])
metrics=pd.DataFrame([
 metric('gas_only_583',out.loc[normal_or_gas,'type'].eq('가스'),out.loc[normal_or_gas,'gas_prediction']),
 metric('shortshot_alone_574',out.loc[normal_or_short,'type'].eq('미성형'),out.loc[normal_or_short,'shortshot_prediction']),
 metric('final_any_defect_591',out.type.ne('정상'),out.final_defect),
 metric('gas_type_full_591',out.type.eq('가스'),out.predicted_type.eq('가스')),
 metric('shortshot_type_full_591',out.type.eq('미성형'),out.predicted_type.eq('미성형'))])
stage=pd.crosstab(out.type,out.predicted_type).reindex(index=['정상','가스','미성형'],columns=['정상','가스','미성형'],fill_value=0)
for obj,name in [(out,'predictions'),(metrics,'metrics'),(pd.DataFrame(audit),'fold_audit'),(out.loc[out.type.ne('정상')],'all_defects')]:
 obj.to_csv(tables/f'rg3_152_gas_shortshot_cascade_{name}.csv',index=False,encoding='utf-8-sig')
stage.to_csv(tables/'rg3_152_gas_shortshot_cascade_type_matrix.csv',encoding='utf-8-sig')
print(metrics.to_string(index=False));print('Type confusion matrix: rows=actual, columns=predicted');print(stage.to_string())
print('Stage 1 gas flagged:',len(out[out.gas_prediction.eq(1)]))
print('Stage 2 newly flagged:',len(out[out.stage2_flag]))


Completed gas full-cohort evaluation 20201021_001626_RG3


Completed gas full-cohort evaluation 20201022_002405_RG3


Completed gas full-cohort evaluation 20201022_044026_RG3


Completed gas full-cohort evaluation 20201023_010128_RG3


Completed gas full-cohort evaluation 20201023_044148_RG3


Original 583 gas predictions reproduced exactly; eight shortshots also genuinely scored.
             evaluation  shots  TP  FP  FN  TN  precision   recall       F1
           gas_only_583    583   8   6   9 560   0.571429 0.470588 0.516129
    shortshot_alone_574    574   5  10   3 556   0.333333 0.625000 0.434783
   final_any_defect_591    591  13  16  12 550   0.448276 0.520000 0.481481
      gas_type_full_591    591   8   6   9 568   0.571429 0.470588 0.516129
shortshot_type_full_591    591   5  10   3 573   0.333333 0.625000 0.434783
Type confusion matrix: rows=actual, columns=predicted
predicted_type   정상  가스  미성형
type                        
정상              550   6   10
가스                9   8    0
미성형               3   0    5
Stage 1 gas flagged: 14
Stage 2 newly flagged: 15
